# ជំពូក ២ (សង្ខេប)៖ បំពង់ផ្ទុកទិន្នន័យ (Data Loading Pipeline)

> **ចំណាំអំពីប្រភព និងអាជ្ញាបណ្ណ** — នេះជា *companion notebook* របស់សហគមន៍ខ្មែរ។
> កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង notebook ផ្លូវការរបស់ឃ្លាំង
> [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/dataloader.ipynb) (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។
> ចំណែកអត្ថបទពន្យល់ជាភាសាខ្មែរគឺជាអត្ថបទដើមរបស់សហគមន៍ (CC BY 4.0) —
> **មិនមែន**ជាការបកប្រែអត្ថបទក្នុងសៀវភៅ *Build a Large Language Model (From Scratch)* ទេ
> ព្រោះអត្ថបទសៀវភៅជាកម្មសិទ្ធិរបស់ Manning Publications និងស្ថិតក្រៅវិសាលភាពអាជ្ញាបណ្ណបើកចំហ។
> សម្រាប់ការពន្យល់ពេញលេញ សូមអានសៀវភៅផ្លូវការ។

**កូដ៖** តក្កវិទ្យាកូដ (logic) ដូច upstream ទាំងស្រុង — មានតែ *comments* ក្នុងកូដដែលត្រូវបានបកប្រែជាភាសាខ្មែរដោយសហគមន៍; កូដដែល comment ជាប់ (commented-out code) នៅដដែល។

**របៀបដំណើរការ៖** ដាក់ notebook នេះក្នុងថតជំពូកពាក់ព័ន្របស់ឃ្លាំង upstream
(ឬចម្លងឯកសារជំនួយដែលកូដត្រូវការ ដូចជា `previous_chapters.py`, `gpt_download.py`
និងឯកសារទិន្នន័យ) រួចដំឡើង dependencies តាម `requirements.txt` ឬប្រើ Google Colab។

## មាតិកា

- ## ០. ចំណាំអំពី notebook សង្ខេប
- ## ១. បំពង់ពេញលេញ

## ០. ចំណាំអំពី notebook សង្ខេប

Notebook នេះសង្ខេបបំពង់ផ្ទុកទិន្នន័យទាំងមូលពីជំពូក ២ ក្នុងកន្លែងតែមួយ ងាយស្រួលយកទៅប្រើ។

In [ ]:
# NBVAL_SKIP
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

## ១. បំពង់ពេញលេញ

ពីការ tokenize ដល់ embedding និង dataloader — ដំណើរការគ្រប់ជំហានជាបន្តបន្ទាប់។

In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # បំបែកអត្ថបទទាំងមូលជា token
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # ប្រើ sliding window ដើម្បីកាត់អត្ថបទជាបន្សំត្រួតគ្នា ប្រវែង max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


def create_dataloader_v1(txt, batch_size, max_length, stride,
                         shuffle=True, drop_last=True, num_workers=0):
    # ចាប់ផ្តើម tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # បង្កើត dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # បង្កើត dataloader
    dataloader = DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

vocab_size = 50257
output_dim = 256
context_length = 1024


token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

batch_size = 8
max_length = 4
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=batch_size,
    max_length=max_length,
    stride=max_length
)

In [ ]:
for batch in dataloader:
    x, y = batch

    token_embeddings = token_embedding_layer(x)
    pos_embeddings = pos_embedding_layer(torch.arange(max_length))

    input_embeddings = token_embeddings + pos_embeddings

    break

In [ ]:
print(input_embeddings.shape)

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ